# 🏦 Loan Underwriting Pipeline — Google Colab Notebook

Run every cell **top-to-bottom**. Each section is clearly numbered.

## 📋 Sections
| # | Section | What it does |
|---|---------|-------------|
| 1 | Environment Setup | Install packages, verify GPU & cuDF |
| 2 | Build File Structure | Create all Python source files on disk |
| 3 | Verify Files on Disk | **List all created files so you can confirm they exist** |
| 4 | Core Pipeline (Person A) | Generate data → Clean → Train model |
| 5 | Google Cloud / BigQuery | GCP auth → Upload clean data |
| 6 | GPU Benchmark | CPU (Pandas) vs GPU (cuDF) benchmark |
| 7 | Person B — GenAI Underwriting | Write files → Set GEMINI key → Run explanations & decisions |
| 8 | End-to-End Integration Test | Full pipeline test |
| 9 | Export Artifacts | Download model + data zip |

> **Before running:**
> 1. `Runtime` → `Change runtime type` → set **Hardware accelerator = GPU** → Save
> 2. Click the 🔑 key icon in the left sidebar → add secret **`GEMINI_API_KEY`** with your key value → enable *Notebook access*

## Section 1 — Environment Setup

In [ ]:
# 1.1 — Verify NVIDIA GPU is assigned
!nvidia-smi

In [ ]:
# 1.2 — Install required Python packages
!pip install -q pynvml python-dotenv google-cloud-bigquery google-genai matplotlib seaborn

In [ ]:
# 1.3 — Verify RAPIDS cuDF pre-installation
try:
    import cudf
    print(f'[OK] cuDF {cudf.__version__} is ready (GPU mode active)')
except ImportError:
    print('[WARN] cuDF not found — ensure GPU runtime is selected')

## Section 2 — Build File Structure on Disk

In [ ]:
# 2.1 — Create directory structure
import os
dirs = ['src/person_a', 'src/person_b', 'tests', 'data']
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('[OK] Directories created:', dirs)

In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/person_a/__init__.py


In [ ]:
%%writefile src/person_b/__init__.py


### 2A — Write Person A source files

In [ ]:
%%writefile src/person_a/column_mapping.py
"""
column_mapping.py - Single source of truth for Kaggle-to-contract field name mapping.

This dictionary is used in clean_data.py on ingestion, so every downstream script
(train.py, scoring_service.py, gpu_benchmark.py) works with the unified contract
field names and never references raw Kaggle column headers ad hoc.
"""

# ---------------------------------------------------------------------
# Kaggle raw column name  ->  Contract / internal field name
# ---------------------------------------------------------------------
KAGGLE_TO_CONTRACT = {
    "MonthlyIncome":                       "income",
    "DebtRatio":                           "debt_ratio",
    "NumberOfOpenCreditLinesAndLoans":      "credit_lines",
    "NumberOfDependents":                   "dependents",
    "NumberOfTimes90DaysLate":              "delinquencies",
}

# Reverse map (contract -> Kaggle) for any script that needs to go the other way
CONTRACT_TO_KAGGLE = {v: k for k, v in KAGGLE_TO_CONTRACT.items()}

# ---------------------------------------------------------------------
# Columns kept but NOT renamed (target + other useful features)
# ---------------------------------------------------------------------
TARGET_COLUMN = "SeriousDlqin2yrs"

# Additional raw features that are kept as-is (not part of the 5-field contract
# but used as model features to improve AUC)
EXTRA_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberRealEstateLoansOrLines",
]

# The 5 contract-name features (after renaming)
CONTRACT_FEATURES = list(KAGGLE_TO_CONTRACT.values())

# All features used for model training (contract names + extras)
ALL_MODEL_FEATURES = CONTRACT_FEATURES + EXTRA_FEATURES


In [ ]:
%%writefile src/person_a/generate_sample_data.py
"""
generate_sample_data.py - Generate a synthetic dataset that mirrors the
"Give Me Some Credit" Kaggle dataset schema and distributions.

This allows the full pipeline to be tested without Kaggle authentication.
Replace data/cs-training.csv with the real Kaggle file when available.

Usage:
    python src/person_a/generate_sample_data.py
"""

import os
import sys
import numpy as np
import pandas as pd

_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
_PROJECT_ROOT = os.path.abspath(os.path.join(_SCRIPT_DIR, "..", ".."))

OUTPUT_PATH = os.path.join(_PROJECT_ROOT, "data", "cs-training.csv")
N_ROWS = 150_000
SEED = 42


def generate():
    rng = np.random.default_rng(SEED)

    # -- Target: ~6.7% default rate (matches Kaggle distribution) ------
    n_default = int(N_ROWS * 0.067)
    target = np.zeros(N_ROWS, dtype=int)
    target[:n_default] = 1
    rng.shuffle(target)

    # -- Features ------------------------------------------------------
    # RevolvingUtilizationOfUnsecuredLines: mostly 0-1, with outlier spikes
    revolving = rng.exponential(0.3, N_ROWS)
    revolving = np.clip(revolving, 0, 50000)
    # Make defaulters have higher utilization on average
    revolving[target == 1] *= rng.uniform(1.5, 3.0, n_default)

    # age: 21-109, median ~52
    age = rng.normal(52, 14, N_ROWS).astype(int)
    age = np.clip(age, 21, 109)

    # NumberOfTime30-59DaysPastDueNotWorse: mostly 0, some 1-2, rare >2
    past_due_30 = rng.poisson(0.2, N_ROWS)
    past_due_30[target == 1] += rng.poisson(0.8, n_default)
    past_due_30 = np.clip(past_due_30, 0, 13)

    # DebtRatio: mostly 0-1, with outlier spikes
    debt_ratio = rng.exponential(0.35, N_ROWS)
    debt_ratio = np.clip(debt_ratio, 0, 5000)

    # MonthlyIncome: median ~5400, with ~20% missing (matching Kaggle)
    monthly_income = rng.lognormal(8.5, 0.8, N_ROWS)
    monthly_income = np.round(monthly_income, 2)
    # Introduce ~20% missing values
    missing_mask = rng.random(N_ROWS) < 0.20
    monthly_income_series = pd.Series(monthly_income, dtype=float)
    monthly_income_series[missing_mask] = np.nan

    # NumberOfOpenCreditLinesAndLoans: 0-58, median ~8
    credit_lines = rng.poisson(8, N_ROWS)
    credit_lines = np.clip(credit_lines, 0, 58)

    # NumberOfTimes90DaysLate: mostly 0
    past_due_90 = rng.poisson(0.1, N_ROWS)
    past_due_90[target == 1] += rng.poisson(0.5, n_default)
    past_due_90 = np.clip(past_due_90, 0, 17)

    # NumberRealEstateLoansOrLines: 0-54, median ~1
    real_estate = rng.poisson(1.0, N_ROWS)
    real_estate = np.clip(real_estate, 0, 54)

    # NumberOfTime60-89DaysPastDueNotWorse: mostly 0
    past_due_60 = rng.poisson(0.08, N_ROWS)
    past_due_60[target == 1] += rng.poisson(0.3, n_default)
    past_due_60 = np.clip(past_due_60, 0, 11)

    # NumberOfDependents: 0-20, with ~2.6% missing
    dependents = rng.poisson(0.8, N_ROWS).astype(float)
    dependents = np.clip(dependents, 0, 20)
    dep_missing = rng.random(N_ROWS) < 0.026
    dependents_series = pd.Series(dependents)
    dependents_series[dep_missing] = np.nan

    # -- Build DataFrame (same column order as Kaggle) -----------------
    df = pd.DataFrame({
        "Unnamed: 0": np.arange(1, N_ROWS + 1),
        "SeriousDlqin2yrs": target,
        "RevolvingUtilizationOfUnsecuredLines": revolving,
        "age": age,
        "NumberOfTime30-59DaysPastDueNotWorse": past_due_30,
        "DebtRatio": debt_ratio,
        "MonthlyIncome": monthly_income_series,
        "NumberOfOpenCreditLinesAndLoans": credit_lines,
        "NumberOfTimes90DaysLate": past_due_90,
        "NumberRealEstateLoansOrLines": real_estate,
        "NumberOfTime60-89DaysPastDueNotWorse": past_due_60,
        "NumberOfDependents": dependents_series,
    })

    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)

    print(f"[generate] DONE - Synthetic dataset saved to {OUTPUT_PATH}")
    print(f"[generate]    Shape: {df.shape}")
    print(f"[generate]    Default rate: {target.mean()*100:.1f}%")
    print(f"[generate]    MonthlyIncome missing: {missing_mask.sum():,} ({missing_mask.mean()*100:.1f}%)")
    print(f"[generate]    NumberOfDependents missing: {dep_missing.sum():,} ({dep_missing.mean()*100:.1f}%)")


if __name__ == "__main__":
    generate()


In [ ]:
%%writefile src/person_a/clean_data.py
"""
clean_data.py - Load the raw Kaggle CSV, apply the central column mapping,
impute missing values, and save a clean dataset.

Usage:
    python -m src.person_a.clean_data
    # or from project root:
    python src/person_a/clean_data.py
"""

import os
import sys
import pandas as pd

# -- Make imports work whether run as a module or as a standalone script --
_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
_PROJECT_ROOT = os.path.abspath(os.path.join(_SCRIPT_DIR, "..", ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from src.person_a.column_mapping import (
    KAGGLE_TO_CONTRACT,
    TARGET_COLUMN,
    ALL_MODEL_FEATURES,
)

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
RAW_CSV = os.path.join(_PROJECT_ROOT, "data", "cs-training.csv")
CLEAN_CSV = os.path.join(_PROJECT_ROOT, "data", "clean_data.csv")


def load_and_clean(raw_path: str = RAW_CSV) -> pd.DataFrame:
    """Load raw Kaggle CSV, rename columns, impute, and return a clean DataFrame."""

    print(f"[clean_data] Loading raw data from {raw_path} ...")
    df = pd.read_csv(raw_path)

    # Drop the unnamed index column that Kaggle ships with the CSV
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)

    # -- Apply the central column mapping ------------------------------
    df.rename(columns=KAGGLE_TO_CONTRACT, inplace=True)
    print(f"[clean_data] Renamed columns: {list(KAGGLE_TO_CONTRACT.keys())} -> {list(KAGGLE_TO_CONTRACT.values())}")

    # -- Impute missing values -----------------------------------------
    # income (was MonthlyIncome): fill with median
    median_income = df["income"].median()
    missing_income = df["income"].isna().sum()
    df["income"] = df["income"].fillna(median_income)
    print(f"[clean_data] Imputed {missing_income} missing 'income' values with median = {median_income:.2f}")

    # dependents (was NumberOfDependents): fill with 0 (mode)
    missing_deps = df["dependents"].isna().sum()
    df["dependents"] = df["dependents"].fillna(0)
    print(f"[clean_data] Imputed {missing_deps} missing 'dependents' values with 0")

    # -- Keep only the columns we need ---------------------------------
    keep_cols = [TARGET_COLUMN] + ALL_MODEL_FEATURES
    # Filter to only columns that actually exist (defensive)
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols]

    # -- Drop any remaining rows with NaN (there shouldn't be many) ----
    before = len(df)
    df.dropna(inplace=True)
    after = len(df)
    if before != after:
        print(f"[clean_data] Dropped {before - after} rows with remaining NaNs")

    print(f"[clean_data] Clean dataset shape: {df.shape}")
    return df


def main():
    df = load_and_clean()
    os.makedirs(os.path.dirname(CLEAN_CSV), exist_ok=True)
    df.to_csv(CLEAN_CSV, index=False)
    print(f"[clean_data] [OK] Saved clean data to {CLEAN_CSV}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/person_a/train.py
"""
train.py - Train a Random Forest classifier on the cleaned loan data.

Targets AUC >= 0.75 on an 80/20 hold-out split.
Saves the trained model to data/model.pkl using joblib.

Usage:
    python -m src.person_a.train
    # or from project root:
    python src/person_a/train.py
"""

import os
import sys
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# -- Make imports work whether run as a module or as a standalone script --
_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
_PROJECT_ROOT = os.path.abspath(os.path.join(_SCRIPT_DIR, "..", ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from src.person_a.column_mapping import TARGET_COLUMN, ALL_MODEL_FEATURES

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
CLEAN_CSV = os.path.join(_PROJECT_ROOT, "data", "clean_data.csv")
MODEL_PATH = os.path.join(_PROJECT_ROOT, "data", "model.pkl")


def train_model(clean_csv: str = CLEAN_CSV) -> tuple:
    """
    Train a RandomForestClassifier and return (model, auc_score).

    Returns:
        model: the trained sklearn RandomForestClassifier
        auc:   ROC AUC score on the hold-out test set
    """
    print(f"[train] Loading clean data from {clean_csv} ...")
    df = pd.read_csv(clean_csv)

    # -- Separate features and target ----------------------------------
    feature_cols = [c for c in ALL_MODEL_FEATURES if c in df.columns]
    X = df[feature_cols]
    y = df[TARGET_COLUMN]

    print(f"[train] Features ({len(feature_cols)}): {feature_cols}")
    print(f"[train] Target distribution:\n{y.value_counts(normalize=True).to_string()}")

    # -- Train/test split (80/20, stratified) --------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    print(f"[train] Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")

    # -- Train Random Forest -------------------------------------------
    # class_weight="balanced" compensates for the heavy class imbalance
    # (~93% non-default, ~7% default).
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    print("[train] Training RandomForestClassifier ...")
    model.fit(X_train, y_train)

    # -- Evaluate ------------------------------------------------------
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    print(f"\n[train] [OK] ROC AUC Score: {auc:.4f}")

    if auc < 0.75:
        print("[train] [WARN]  AUC is below 0.75 - consider tuning hyperparameters.")
    else:
        print("[train] [OK] AUC target (>= 0.75) met!")

    # -- Feature importance --------------------------------------------
    importances = sorted(
        zip(feature_cols, model.feature_importances_),
        key=lambda x: x[1],
        reverse=True,
    )
    print("\n[train] Feature importances (top 10):")
    for name, imp in importances[:10]:
        print(f"  {name:45s} {imp:.4f}")

    return model, auc


def main():
    model, auc = train_model()

    # -- Save model ----------------------------------------------------
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    joblib.dump(model, MODEL_PATH)
    print(f"\n[train] [OK] Model saved to {MODEL_PATH}")
    print(f"[train] Final AUC: {auc:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/person_a/scoring_service.py
"""
scoring_service.py - Production-ready API contract functions for Person B's
Streamlit dashboard.

Functions:
    score_applicant(applicant_dict)       -> float  (0.0 – 1.0 risk probability)
    get_cohort_default_rate(applicant_dict) -> float  (0.0 – 100.0 percentage)

Contract input dict keys:
    income, debt_ratio, credit_lines, delinquencies, dependents
"""

import os
import sys
import json
import time
import joblib
import numpy as np
import pandas as pd

# -- Make imports work whether run as a module or as a standalone script --
_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
_PROJECT_ROOT = os.path.abspath(os.path.join(_SCRIPT_DIR, "..", ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(_PROJECT_ROOT, ".env"))

from src.person_a.column_mapping import ALL_MODEL_FEATURES, CONTRACT_FEATURES

# ---------------------------------------------------------------------
# Paths & Configuration
# ---------------------------------------------------------------------
MODEL_PATH = os.path.join(_PROJECT_ROOT, "data", "model.pkl")
CLEAN_CSV = os.path.join(_PROJECT_ROOT, "data", "clean_data.csv")
CACHE_PATH = os.path.join(_PROJECT_ROOT, "data", "cohort_cache.json")

BQ_DATASET = os.environ.get("BQ_DATASET", "loan_underwriting")
BQ_TABLE = os.environ.get("BQ_TABLE", "loan_applicants")
BQ_QUERY_TIMEOUT_SECONDS = float(os.environ.get("BQ_QUERY_TIMEOUT", "5.0"))

# ---------------------------------------------------------------------
# Model loading (cached at module level for performance)
# ---------------------------------------------------------------------
_model = None


def _get_model():
    """Lazy-load the trained model (cached after first call)."""
    global _model
    if _model is None:
        if not os.path.exists(MODEL_PATH):
            raise FileNotFoundError(
                f"Model file not found at {MODEL_PATH}. Run train.py first."
            )
        _model = joblib.load(MODEL_PATH)
    return _model


# ---------------------------------------------------------------------
# Default / median values for features not in the 5-field contract
# These are computed from the training data and hard-coded here for
# inference speed; they only need updating if the dataset changes.
# ---------------------------------------------------------------------
_FEATURE_DEFAULTS = {
    # Contract features (will be overridden by applicant_dict)
    "income": 5400.0,
    "debt_ratio": 0.35,
    "credit_lines": 8,
    "delinquencies": 0,
    "dependents": 0,
    # Extra features (median/mode defaults from training data)
    "RevolvingUtilizationOfUnsecuredLines": 0.15,
    "age": 52,
    "NumberOfTime30-59DaysPastDueNotWorse": 0,
    "NumberOfTime60-89DaysPastDueNotWorse": 0,
    "NumberRealEstateLoansOrLines": 1,
}


# =====================================================================
# FUNCTION 1: score_applicant
# =====================================================================
def score_applicant(applicant_dict: dict) -> float:
    """
    Predict the probability of default for a single applicant.

    Args:
        applicant_dict: dict with keys
            {income, debt_ratio, credit_lines, delinquencies, dependents}

    Returns:
        float between 0.0 (low risk) and 1.0 (high risk)
    """
    model = _get_model()

    # Build a feature vector in the order the model expects
    row = dict(_FEATURE_DEFAULTS)  # start with defaults
    for key in CONTRACT_FEATURES:
        if key in applicant_dict:
            row[key] = applicant_dict[key]

    # Create a single-row DataFrame with the exact feature order
    feature_order = ALL_MODEL_FEATURES
    df = pd.DataFrame([row])[feature_order]

    # Predict probability of class 1 (default)
    prob = model.predict_proba(df)[0, 1]
    return float(np.clip(prob, 0.0, 1.0))


# =====================================================================
# FUNCTION 2: get_cohort_default_rate
# =====================================================================
def get_cohort_default_rate(applicant_dict: dict) -> float:
    """
    Query BigQuery for the historical default rate of applicants in a
    similar cohort (income +-15%, same credit_lines).

    Falls back to wider income bands (+-30%, +-50%) or the overall population
    default rate if the cohort is too small (< 20 rows). Falls back to a
    pre-computed local cache if BigQuery is unreachable or times out.

    NOTE: Cohort rates are drawn from the same BigQuery table used for
    model training, so a test applicant present in training data could
    overlap with their own cohort.  Not a blocker for a hackathon demo
    but worth noting for a production system.

    Args:
        applicant_dict: dict with keys {income, credit_lines, ...}

    Returns:
        float between 0.0 and 100.0 (percentage)
    """
    income = applicant_dict.get("income", 5400.0)
    credit_lines = applicant_dict.get("credit_lines", 8)

    # -- Try live BigQuery query ---------------------------------------
    try:
        rate = _query_bigquery_cohort(income, credit_lines)
        if rate is not None:
            return rate
    except Exception as e:
        print(f"[scoring] [WARN]  BigQuery query failed: {e}")

    # -- Fallback: local cache -----------------------------------------
    cached_rate = _lookup_cache(applicant_dict)
    if cached_rate is not None:
        print("[scoring] Using cached cohort default rate (BigQuery unavailable)")
        return cached_rate

    # -- Fallback: compute from local CSV ------------------------------
    print("[scoring] Computing cohort rate from local CSV fallback")
    return _compute_local_cohort_rate(income, credit_lines)


def _query_bigquery_cohort(
    income: float, credit_lines: int, min_cohort: int = 20
) -> float | None:
    """
    Query BigQuery with progressively wider income bands until the cohort
    has >= min_cohort rows.  Returns the default rate (0–100), or None if
    BigQuery is unreachable.
    """
    from google.cloud import bigquery

    gcp_project_id = os.environ.get("GCP_PROJECT_ID")
    if not gcp_project_id:
        raise KeyError(
            "GCP_PROJECT_ID environment variable is missing or empty. Please set it in your .env file."
        )

    client = bigquery.Client(project=gcp_project_id)
    table_ref = f"`{gcp_project_id}.{BQ_DATASET}.{BQ_TABLE}`"

    # Progressive income bands: +-15%, +-30%, +-50%, then full population
    bands = [0.15, 0.30, 0.50]

    for band in bands:
        lo = income * (1 - band)
        hi = income * (1 + band)

        query = f"""
        SELECT
            COUNT(*) AS cohort_size,
            COUNTIF(SeriousDlqin2yrs = 1) AS default_count
        FROM {table_ref}
        WHERE income BETWEEN {lo} AND {hi}
          AND credit_lines = {credit_lines}
        """

        job_config = bigquery.QueryJobConfig(
            query_parameters=[],
        )
        query_job = client.query(query, job_config=job_config)

        # Apply timeout
        try:
            rows = list(query_job.result(timeout=BQ_QUERY_TIMEOUT_SECONDS))
        except Exception as e:
            print(f"[scoring] BigQuery timeout/error at +-{int(band*100)}% band: {e}")
            return None

        if rows and rows[0]["cohort_size"] >= min_cohort:
            cohort_size = rows[0]["cohort_size"]
            default_count = rows[0]["default_count"]
            rate = (default_count / cohort_size) * 100.0
            print(
                f"[scoring] Cohort found: {cohort_size} rows "
                f"(income +-{int(band*100)}%), default rate = {rate:.2f}%"
            )
            return round(rate, 2)

        print(
            f"[scoring] Cohort too small at +-{int(band*100)}% "
            f"({rows[0]['cohort_size'] if rows else 0} rows), widening..."
        )

    # -- Full population fallback --------------------------------------
    query = f"""
    SELECT
        COUNT(*) AS total,
        COUNTIF(SeriousDlqin2yrs = 1) AS default_count
    FROM {table_ref}
    """
    rows = list(client.query(query).result(timeout=BQ_QUERY_TIMEOUT_SECONDS))
    if rows and rows[0]["total"] > 0:
        rate = (rows[0]["default_count"] / rows[0]["total"]) * 100.0
        print(f"[scoring] Using full population default rate: {rate:.2f}%")
        return round(rate, 2)

    return None


# ---------------------------------------------------------------------
# Local fallback: compute from clean_data.csv
# ---------------------------------------------------------------------
def _compute_local_cohort_rate(
    income: float, credit_lines: int, min_cohort: int = 20
) -> float:
    """Compute cohort default rate from the local CSV as a fallback."""
    if not os.path.exists(CLEAN_CSV):
        print("[scoring] [WARN]  Local CSV not found; returning population estimate.")
        return 6.7  # approximate overall default rate for this dataset

    df = pd.read_csv(CLEAN_CSV)

    bands = [0.15, 0.30, 0.50]
    for band in bands:
        lo = income * (1 - band)
        hi = income * (1 + band)
        cohort = df[
            (df["income"].between(lo, hi)) & (df["credit_lines"] == credit_lines)
        ]
        if len(cohort) >= min_cohort:
            rate = cohort["SeriousDlqin2yrs"].mean() * 100.0
            return round(rate, 2)

    # Full population fallback
    rate = df["SeriousDlqin2yrs"].mean() * 100.0
    return round(rate, 2)


# ---------------------------------------------------------------------
# Demo cache: pre-computed cohort rates for the 3 demo profiles
# ---------------------------------------------------------------------
def precompute_demo_cache():
    """
    Pre-compute and save cohort default rates for the 3 demo profiles.
    Run this once before the demo to prime the local cache fallback.
    """
    demo_profiles = {
        "low_risk": {"income": 9500, "debt_ratio": 0.15, "credit_lines": 10,
                     "delinquencies": 0, "dependents": 1},
        "high_risk": {"income": 2200, "debt_ratio": 0.85, "credit_lines": 3,
                      "delinquencies": 4, "dependents": 3},
        "borderline": {"income": 5000, "debt_ratio": 0.45, "credit_lines": 7,
                       "delinquencies": 1, "dependents": 2},
    }

    cache = {}
    for label, profile in demo_profiles.items():
        rate = _compute_local_cohort_rate(
            profile["income"], profile["credit_lines"]
        )
        score = score_applicant(profile)
        cache[label] = {
            "profile": profile,
            "cohort_default_rate": rate,
            "risk_score": round(score, 4),
        }
        print(f"[cache] {label}: risk_score={score:.4f}, cohort_rate={rate:.2f}%")

    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    with open(CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)
    print(f"[cache] [OK] Saved demo cache to {CACHE_PATH}")
    return cache


def _lookup_cache(applicant_dict: dict) -> float | None:
    """Look up an applicant in the pre-computed demo cache."""
    if not os.path.exists(CACHE_PATH):
        return None
    try:
        with open(CACHE_PATH) as f:
            cache = json.load(f)
        for label, entry in cache.items():
            profile = entry["profile"]
            if (
                abs(profile["income"] - applicant_dict.get("income", -1)) < 1.0
                and profile["credit_lines"] == applicant_dict.get("credit_lines", -1)
            ):
                print(f"[scoring] Cache hit: {label}")
                return entry["cohort_default_rate"]
    except Exception:
        pass
    return None


In [ ]:
%%writefile src/person_a/gpu_benchmark.py
"""
gpu_benchmark.py - CPU (Pandas) vs GPU (cuDF) benchmark for data processing.

Scales the clean dataset incrementally (150K -> 1.5M -> test memory headroom
before attempting 15M) and measures runtime for typical feature-engineering
operations.

If the GPU runs out of memory at a given scale, the script caps the benchmark
at whatever size completed successfully and documents the cap in the chart.

IMPORTANT: This script is designed to run inside WSL2 Ubuntu with a local
NVIDIA GPU (compute capability 7.0+) and RAPIDS installed via conda/miniforge.
It will NOT work on Windows natively or in Google Colab.

Usage (inside WSL2 with rapids-env activated):
    python -m src.person_a.gpu_benchmark
"""

import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for WSL2 / headless
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -- Project root ------------------------------------------------------
_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
_PROJECT_ROOT = os.path.abspath(os.path.join(_SCRIPT_DIR, "..", ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

CLEAN_CSV = os.path.join(_PROJECT_ROOT, "data", "clean_data.csv")
CHART_PATH = os.path.join(_PROJECT_ROOT, "data", "cpu_vs_gpu_benchmark.png")

# ---------------------------------------------------------------------
# GPU memory & cuDF availability check
# ---------------------------------------------------------------------
CUDF_AVAILABLE = False
GPU_MEM_MB = 0

try:
    import cudf  # noqa: F401
    CUDF_AVAILABLE = True
    # Try to read GPU memory via nvidia-smi or pynvml
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        GPU_MEM_MB = info.total // (1024 * 1024)
        gpu_name = pynvml.nvmlDeviceGetName(handle)
        if isinstance(gpu_name, bytes):
            gpu_name = gpu_name.decode()
        pynvml.nvmlShutdown()
        print(f"[benchmark] GPU detected: {gpu_name} ({GPU_MEM_MB:,} MB VRAM)", flush=True)
    except Exception:
        # Fallback: assume 8GB if we can't query
        GPU_MEM_MB = 8192
        print(f"[benchmark] cuDF available but couldn't query VRAM. Assuming {GPU_MEM_MB} MB.", flush=True)
except ImportError:
    print("[benchmark] [WARN]  cuDF not available. Running CPU-only benchmark.", flush=True)
    print("[benchmark]    Install RAPIDS via conda in WSL2 for GPU benchmarks.", flush=True)


# ---------------------------------------------------------------------
# Benchmark workload: typical feature-engineering operations
# ---------------------------------------------------------------------
def run_workload_pandas(df: pd.DataFrame) -> None:
    """Simulate typical feature-engineering operations using Pandas (CPU)."""
    # 1. Compute debt-to-income ratio (bounded)
    df["dti_ratio"] = df["debt_ratio"] * df["income"]
    # 2. Binning income into brackets
    df["income_bracket"] = pd.cut(df["income"], bins=10, labels=False)
    # 3. Group-by aggregation: mean delinquencies per income bracket
    _ = df.groupby("income_bracket")["delinquencies"].mean()
    # 4. Sort by risk-relevant column
    _ = df.sort_values("debt_ratio", ascending=False)
    # 5. Rolling / cumulative sum (simulate sequential scan)
    _ = df["income"].cumsum()


def run_workload_cudf(gdf) -> None:
    """Same workload using cuDF (GPU)."""
    import cudf as _cudf  # noqa: F811
    # 1. Compute debt-to-income
    gdf["dti_ratio"] = gdf["debt_ratio"] * gdf["income"]
    # 2. Binning income
    gdf["income_bracket"] = _cudf.cut(gdf["income"], bins=10, labels=False)
    # 3. Group-by aggregation
    _ = gdf.groupby("income_bracket")["delinquencies"].mean()
    # 4. Sort
    _ = gdf.sort_values("debt_ratio", ascending=False)
    # 5. Cumulative sum
    _ = gdf["income"].cumsum()


# ---------------------------------------------------------------------
# Synthetic data generation
# ---------------------------------------------------------------------
def scale_up(df: pd.DataFrame, target_rows: int, seed: int = 42) -> pd.DataFrame:
    """
    Repeat the base dataframe to reach `target_rows`, adding small
    random noise to numeric columns so it's not a trivial duplicate.
    """
    rng = np.random.default_rng(seed)
    repeats = (target_rows // len(df)) + 1
    scaled = pd.concat([df] * repeats, ignore_index=True).head(target_rows)
    # Add +-5% noise to income and debt_ratio
    if "income" in scaled.columns:
        noise = rng.uniform(0.95, 1.05, size=len(scaled))
        scaled["income"] = scaled["income"] * noise
    if "debt_ratio" in scaled.columns:
        noise = rng.uniform(0.95, 1.05, size=len(scaled))
        scaled["debt_ratio"] = scaled["debt_ratio"] * noise
    return scaled


# ---------------------------------------------------------------------
# Benchmark runner
# ---------------------------------------------------------------------
def benchmark_pandas(df: pd.DataFrame) -> float:
    """Time the Pandas workload. Returns seconds."""
    df_copy = df.copy()
    start = time.perf_counter()
    run_workload_pandas(df_copy)
    return time.perf_counter() - start


def benchmark_cudf(df: pd.DataFrame) -> float | None:
    """
    Time the cuDF workload. Returns seconds, or None if OOM / not available.
    """
    if not CUDF_AVAILABLE:
        return None
    import cudf as _cudf  # noqa: F811
    try:
        gdf = _cudf.from_pandas(df)
        # Warm-up pass (first transfer is slower)
        run_workload_cudf(gdf.copy())
        # Timed pass
        start = time.perf_counter()
        run_workload_cudf(gdf.copy())
        elapsed = time.perf_counter() - start
        del gdf
        return elapsed
    except Exception as e:
        print(f"[benchmark] [WARN]  cuDF failed: {e}")
        return None


def estimate_max_rows(base_rows: int, base_mem_mb: float, gpu_mem_mb: int) -> int:
    """
    Estimate the max row count that fits in GPU memory, reserving 30% headroom.
    base_mem_mb is a rough estimate of how much VRAM the base dataset uses.
    """
    usable_mb = gpu_mem_mb * 0.70  # keep 30% headroom
    if base_mem_mb <= 0:
        return 15_000_000  # fallback: assume it fits
    max_rows = int((usable_mb / base_mem_mb) * base_rows)
    return max_rows


def main():
    if not os.path.exists(CLEAN_CSV):
        print(f"[benchmark] [WARN]  Clean data not found at {CLEAN_CSV}", flush=True)
        print("[benchmark]    Run clean_data.py first.", flush=True)
        sys.exit(1)

    base_df = pd.read_csv(CLEAN_CSV)
    base_rows = len(base_df)
    base_mem_mb = base_df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"[benchmark] Base dataset: {base_rows:,} rows, ~{base_mem_mb:.1f} MB in-memory", flush=True)

    # -- Define benchmark scales ---------------------------------------
    # Start with the intended scales, then cap based on GPU memory
    intended_scales = [base_rows, 1_500_000, 15_000_000]

    if CUDF_AVAILABLE and GPU_MEM_MB > 0:
        max_rows = estimate_max_rows(base_rows, base_mem_mb, GPU_MEM_MB)
        print(f"[benchmark] Estimated max GPU rows (with 30% headroom): {max_rows:,}", flush=True)
        # Cap the largest scale
        capped_scales = []
        for s in intended_scales:
            if s <= max_rows:
                capped_scales.append(s)
            else:
                # Use the max that fits, but only if it's larger than what we have
                if not capped_scales or max_rows > capped_scales[-1]:
                    capped_scales.append(max_rows)
                break
        scales = capped_scales
    else:
        scales = intended_scales

    print(f"[benchmark] Benchmark scales: {[f'{s:,}' for s in scales]}", flush=True)

    # -- Run benchmarks ------------------------------------------------
    results = []
    actual_max_gpu_scale = 0

    for scale in scales:
        print(f"\n{'='*60}", flush=True)
        print(f"  Scale: {scale:,} rows", flush=True)
        print(f"{'='*60}", flush=True)

        df_scaled = scale_up(base_df, scale)

        # CPU (Pandas)
        cpu_time = benchmark_pandas(df_scaled)
        print(f"  CPU (Pandas):  {cpu_time:.3f}s", flush=True)

        # GPU (cuDF)
        gpu_time = benchmark_cudf(df_scaled)
        if gpu_time is not None:
            speedup = cpu_time / gpu_time if gpu_time > 0 else float("inf")
            print(f"  GPU (cuDF):    {gpu_time:.3f}s  ({speedup:.1f}x speedup)", flush=True)
            actual_max_gpu_scale = scale
        else:
            print(f"  GPU (cuDF):    SKIPPED (OOM or not available)", flush=True)

        results.append({
            "rows": scale,
            "cpu_seconds": cpu_time,
            "gpu_seconds": gpu_time,
        })

        del df_scaled

    # -- Generate chart ------------------------------------------------
    generate_chart(results, actual_max_gpu_scale)
    print(f"\n[benchmark] [OK] Benchmark complete. Chart saved to {CHART_PATH}", flush=True)


def generate_chart(results: list[dict], actual_max_gpu_scale: int):
    """Generate a grouped bar chart comparing CPU vs GPU times."""
    fig, ax = plt.subplots(figsize=(12, 7))

    labels = [f"{r['rows']:,}" for r in results]
    cpu_times = [r["cpu_seconds"] for r in results]
    gpu_times = [r["gpu_seconds"] if r["gpu_seconds"] is not None else 0 for r in results]

    x = np.arange(len(labels))
    width = 0.35

    bars_cpu = ax.bar(x - width / 2, cpu_times, width, label="CPU (Pandas)",
                      color="#3b82f6", edgecolor="white", linewidth=0.5)
    bars_gpu = ax.bar(x + width / 2, gpu_times, width, label="GPU (cuDF / RAPIDS)",
                      color="#10b981", edgecolor="white", linewidth=0.5)

    # Annotate bars with times
    for bar, t in zip(bars_cpu, cpu_times):
        ax.annotate(f"{t:.2f}s", xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 5), textcoords="offset points", ha="center", fontsize=9)
    for bar, t, r in zip(bars_gpu, gpu_times, results):
        if r["gpu_seconds"] is not None:
            speedup = r["cpu_seconds"] / r["gpu_seconds"] if r["gpu_seconds"] > 0 else 0
            ax.annotate(f"{t:.2f}s\n({speedup:.1f}x)",
                        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        xytext=(0, 5), textcoords="offset points", ha="center", fontsize=9)
        else:
            ax.annotate("N/A",
                        xy=(bar.get_x() + bar.get_width() / 2, 0.01),
                        xytext=(0, 5), textcoords="offset points", ha="center",
                        fontsize=9, color="gray")

    ax.set_xlabel("Dataset Size (rows)", fontsize=12)
    ax.set_ylabel("Execution Time (seconds)", fontsize=12)

    # Build title with actual cap info
    cap_note = ""
    if actual_max_gpu_scale > 0 and actual_max_gpu_scale < 15_000_000:
        cap_note = f"\n(GPU benchmark capped at {actual_max_gpu_scale:,} rows due to VRAM limits)"
    ax.set_title(
        f"CPU vs GPU: Feature Engineering Benchmark{cap_note}",
        fontsize=14, fontweight="bold"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    os.makedirs(os.path.dirname(CHART_PATH), exist_ok=True)
    plt.savefig(CHART_PATH, dpi=150, bbox_inches="tight")
    plt.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/person_a/cloud_upload.py
"""
cloud_upload.py - Load data/clean_data.csv directly into BigQuery.

Prerequisites:
    - Google Cloud SDK authenticated (gcloud auth application-default login)
      OR a service account JSON key set via GOOGLE_APPLICATION_CREDENTIALS env var.
    - A GCP project with the BigQuery API enabled.
    - GCP_PROJECT_ID set in the project .env file.

Usage:
    python -m src.person_a.cloud_upload
    # or:
    python src/person_a/cloud_upload.py
"""

import os
import sys

# -- Make imports work whether run as a module or as a standalone script --
_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
_PROJECT_ROOT = os.path.abspath(os.path.join(_SCRIPT_DIR, "..", ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

# Load environment variables from .env
from dotenv import load_dotenv

load_dotenv(os.path.join(_PROJECT_ROOT, ".env"))

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
CLEAN_CSV = os.path.join(_PROJECT_ROOT, "data", "clean_data.csv")
BQ_DATASET = "loan_underwriting"
BQ_TABLE = "loan_applicants"


def main():
    # ------------------------------------------------------------------
    # 1. Validate prerequisites
    # ------------------------------------------------------------------
    gcp_project_id = os.environ.get("GCP_PROJECT_ID")
    if not gcp_project_id:
        print(
            "[cloud] [ERROR] GCP_PROJECT_ID is not set.\n"
            "       Add it to your .env file or export it:\n"
            "         GCP_PROJECT_ID=my-project-id"
        )
        sys.exit(1)

    if not os.path.exists(CLEAN_CSV):
        print(
            f"[cloud] [ERROR] CSV file not found at {CLEAN_CSV}\n"
            "       Run clean_data.py first to generate it."
        )
        sys.exit(1)

    # ------------------------------------------------------------------
    # 2. Upload to BigQuery
    # ------------------------------------------------------------------
    try:
        
        from google.cloud import bigquery
        from google.api_core.exceptions import NotFound

        client = bigquery.Client(project=gcp_project_id)

        # -- Ensure dataset exists -------------------------------------
        dataset_ref = client.dataset(BQ_DATASET)
        try:
            client.get_dataset(dataset_ref)
            print(f"[cloud] Using existing BigQuery dataset: {BQ_DATASET}")
        except NotFound:
            print(f"[cloud] Creating BigQuery dataset: {BQ_DATASET}")
            dataset = bigquery.Dataset(dataset_ref)
            dataset.location = "US"
            client.create_dataset(dataset)

        # -- Load CSV directly into BigQuery ---------------------------
        table_id = f"{gcp_project_id}.{BQ_DATASET}.{BQ_TABLE}"

        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            write_disposition="WRITE_TRUNCATE",
        )

        print(f"[cloud] Loading {CLEAN_CSV} -> BigQuery table {table_id} ...")

        with open(CLEAN_CSV, "rb") as csv_file:
            load_job = client.load_table_from_file(
                csv_file, table_id, job_config=job_config
            )

        # Wait for the load job to complete
        load_job.result()

        # Confirm row count
        table = client.get_table(table_id)
        print(f"[cloud] [OK] Loaded {table.num_rows:,} rows into {table_id}")

    except FileNotFoundError as e:
        print(f"[cloud] [ERROR] File not found: {e}")
        sys.exit(1)

    except ImportError:
        print(
            "[cloud] [ERROR] google-cloud-bigquery is not installed.\n"
            "       Run: pip install google-cloud-bigquery"
        )
        sys.exit(1)

    except Exception as e:
        error_msg = str(e).lower()
        if "credentials" in error_msg or "authentication" in error_msg or "auth" in error_msg:
            print(
                f"[cloud] [ERROR] Authentication failed: {e}\n"
                "       Run: gcloud auth application-default login\n"
                "       Or set GOOGLE_APPLICATION_CREDENTIALS to your service account key."
            )
        else:
            print(f"[cloud] [ERROR] BigQuery upload failed: {e}")
        sys.exit(1)

    print("\n[cloud] [OK] Cloud upload complete.")


if __name__ == "__main__":
    main()


### 2B — Write Person B source files

In [ ]:
%%writefile src/person_b/fake_applicants.py
# src/person_b/fake_applicants.py

fake_applicants = [
    {
        "income": 12500,
        "debt_ratio": 0.15,
        "credit_lines": 8,
        "delinquencies": 0,
        "dependents": 0,
        "risk_level_label": "Low"
    },
    {
        "income": 22000,
        "debt_ratio": 0.08,
        "credit_lines": 12,
        "delinquencies": 0,
        "dependents": 2,
        "risk_level_label": "Low"
    },
    {
        "income": 6000,
        "debt_ratio": 0.35,
        "credit_lines": 6,
        "delinquencies": 0,
        "dependents": 1,
        "risk_level_label": "Medium"
    },
    {
        "income": 4500,
        "debt_ratio": 0.48,
        "credit_lines": 5,
        "delinquencies": 0,
        "dependents": 3,
        "risk_level_label": "Medium"
    },
    {
        "income": 3200,
        "debt_ratio": 0.22,
        "credit_lines": 3,
        "delinquencies": 0,
        "dependents": 0,
        "risk_level_label": "Medium"
    },
    {
        "income": 2500,
        "debt_ratio": 0.65,
        "credit_lines": 4,
        "delinquencies": 2,
        "dependents": 1,
        "risk_level_label": "High"
    },
    {
        "income": 5000,
        "debt_ratio": 0.55,
        "credit_lines": 15,
        "delinquencies": 3,
        "dependents": 4,
        "risk_level_label": "High"
    },
    {
        "income": 1500,
        "debt_ratio": 1.20,
        "credit_lines": 10,
        "delinquencies": 5,
        "dependents": 2,
        "risk_level_label": "High"
    }
]

if __name__ == "__main__":
    import pprint
    print("Generated 8 fake applicant records:")
    pprint.pprint(fake_applicants)


In [ ]:
%%writefile src/person_b/explain_api.py
# src/person_b/explain_api.py
import os
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY"),
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=5,
            initial_delay=2.0,
            max_delay=15.0,
            http_status_codes=[429, 500, 502, 503, 504]
        )
    )
)
MODEL = "gemini-2.5-flash-lite"


def explain_risk(applicant: dict, risk_score: float, cohort_rate: float) -> str:
    """Call Gemini to produce a 1-2 sentence plain-language risk explanation."""
    prompt = (
        "You are a loan underwriting analyst. Given the applicant data below, "
        "write exactly 1-2 sentences explaining why this applicant presents "
        "the indicated level of default risk. Be specific about which factors "
        "drive the risk assessment. Do not use bullet points or headers.\n\n"
        f"Applicant data:\n"
        f"  Monthly income: ${applicant['income']:,}\n"
        f"  Debt ratio: {applicant['debt_ratio']:.0%}\n"
        f"  Open credit lines: {applicant['credit_lines']}\n"
        f"  Past delinquencies (90+ days): {applicant['delinquencies']}\n"
        f"  Number of dependents: {applicant['dependents']}\n\n"
        f"ML-predicted default risk score: {risk_score:.2f} (0 = no risk, 1 = certain default)\n"
        f"Cohort default rate: {cohort_rate:.1f}%\n"
    )
    try:
        response = client.models.generate_content(model=MODEL, contents=prompt)
        return response.text.strip()
    except Exception as e:
        return f"[Gemini unavailable] Risk score is {risk_score:.2f}. Error: {e}"


# --------------- test driver ---------------
if __name__ == "__main__":
    import sys
    import os
    sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

    from person_b.fake_applicants import fake_applicants

    # Made-up risk_score and cohort_rate values matched to risk labels
    test_params = [
        (0.05, 3.2),   # Low
        (0.08, 2.8),   # Low
        (0.32, 8.5),   # Medium
        (0.41, 11.0),  # Medium
        (0.28, 7.1),   # Medium
        (0.72, 18.4),  # High
        (0.68, 16.9),  # High
        (0.91, 24.5),  # High
    ]

    for i, (applicant, (score, cohort)) in enumerate(zip(fake_applicants, test_params)):
        label = applicant.get("risk_level_label", "?")
        print(f"\n--- Applicant {i+1} ({label} risk) ---")
        print(f"  Income=${applicant['income']:,}  Debt={applicant['debt_ratio']:.0%}  "
              f"Delinq={applicant['delinquencies']}  Score={score}")
        explanation = explain_risk(applicant, score, cohort)
        print(f"  Explanation: {explanation}")
        # Small delay to respect free-tier rate limits
        time.sleep(4)


In [ ]:
%%writefile src/person_b/decision_engine.py
# src/person_b/decision_engine.py
import os
import json
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY"),
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=5,
            initial_delay=2.0,
            max_delay=15.0,
            http_status_codes=[429, 500, 502, 503, 504]
        )
    )
)
MODEL = "gemini-2.5-flash-lite"

VALID_DECISIONS = {"Approve", "Manual Review", "Decline"}
VALID_CONFIDENCES = {"High", "Medium", "Low"}


def _fallback_from_explanation(explanation: str) -> dict:
    """Keyword-based heuristic when Gemini is unavailable."""
    text = explanation.lower()
    if any(w in text for w in ["extremely high", "very high", "unsustainable", "severe"]):
        return {"recommendation": "Decline", "confidence": "High"}
    if any(w in text for w in ["high risk", "high default"]):
        return {"recommendation": "Decline", "confidence": "Medium"}
    if any(w in text for w in ["moderate", "elevated", "borderline"]):
        return {"recommendation": "Manual Review", "confidence": "Medium"}
    if any(w in text for w in ["very low", "excellent", "exceptionally"]):
        return {"recommendation": "Approve", "confidence": "High"}
    if any(w in text for w in ["low risk", "low default"]):
        return {"recommendation": "Approve", "confidence": "Medium"}
    return {"recommendation": "Manual Review", "confidence": "Low"}


def recommend_decision(explanation: str) -> dict:
    """Call Gemini to produce a structured underwriting decision from a risk explanation.

    Returns: {"recommendation": "Approve"|"Manual Review"|"Decline",
              "confidence": "High"|"Medium"|"Low"}
    """
    prompt = (
        "You are a senior loan underwriter. Based on the risk explanation below, "
        "output a JSON object with exactly two keys:\n"
        '  "recommendation": one of "Approve", "Manual Review", or "Decline"\n'
        '  "confidence": one of "High", "Medium", or "Low"\n\n'
        "Return ONLY the raw JSON object, no markdown, no commentary.\n\n"
        f"Risk explanation:\n{explanation}"
    )
    try:
        response = client.models.generate_content(model=MODEL, contents=prompt)
        raw = response.text.strip()
        # Strip markdown fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        result = json.loads(raw)
        # Validate fields
        if result.get("recommendation") not in VALID_DECISIONS:
            raise ValueError(f"Bad recommendation: {result.get('recommendation')}")
        if result.get("confidence") not in VALID_CONFIDENCES:
            raise ValueError(f"Bad confidence: {result.get('confidence')}")
        return result
    except Exception as e:
        print(f"  [Gemini fallback] {e}")
        return _fallback_from_explanation(explanation)


# --------------- test driver ---------------
if __name__ == "__main__":
    import sys
    import os
    sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

    from person_b.fake_applicants import fake_applicants
    from person_b.explain_api import explain_risk

    test_params = [
        (0.05, 3.2),   # Low
        (0.08, 2.8),   # Low
        (0.32, 8.5),   # Medium
        (0.41, 11.0),  # Medium
        (0.28, 7.1),   # Medium
        (0.72, 18.4),  # High
        (0.68, 16.9),  # High
        (0.91, 24.5),  # High
    ]

    for i, (applicant, (score, cohort)) in enumerate(zip(fake_applicants, test_params)):
        label = applicant.get("risk_level_label", "?")
        print(f"\n=== Applicant {i+1} ({label} risk) ===")

        explanation = explain_risk(applicant, score, cohort)
        print(f"  Explanation: {explanation[:120]}...")

        decision = recommend_decision(explanation)
        print(f"  Decision:    {json.dumps(decision)}")
        time.sleep(4)


### 2C — Write integration test file

In [ ]:
%%writefile tests/test_full_chain.py
"""
test_full_chain.py - End-to-end integration test for the loan underwriting pipeline.

Runs 2 sample applicants through:
  score_applicant() -> get_cohort_default_rate() -> explain_risk() -> recommend_decision()

Prints every intermediate output with its Python type so any mismatch is
immediately visible.

Usage (from project root):
  python tests/test_full_chain.py
"""

import sys
import os
import time

# Ensure project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.insert(0, PROJECT_ROOT)

from src.person_a.scoring_service import score_applicant, get_cohort_default_rate
from src.person_b.explain_api import explain_risk
from src.person_b.decision_engine import recommend_decision

# Two sample applicants from fake_applicants.py (one low risk, one high risk)
SAMPLE_APPLICANTS = [
    {
        "income": 12500,
        "debt_ratio": 0.15,
        "credit_lines": 8,
        "delinquencies": 0,
        "dependents": 0,
        "label": "Low Risk",
    },
    {
        "income": 1500,
        "debt_ratio": 1.20,
        "credit_lines": 10,
        "delinquencies": 5,
        "dependents": 2,
        "label": "High Risk",
    },
]


def run_chain(applicant: dict) -> None:
    """Run one applicant through the full pipeline, printing every step."""
    label = applicant.pop("label", "Unknown")
    print(f"\n{'='*60}")
    print(f"  Applicant: {label}")
    print(f"  Input dict: {applicant}")
    print(f"{'='*60}")

    # Step 1: score_applicant
    risk_score = score_applicant(applicant)
    print(f"\n  [1] score_applicant()")
    print(f"      Return value : {risk_score}")
    print(f"      Type         : {type(risk_score).__name__}")
    print(f"      In range 0-1?: {0.0 <= risk_score <= 1.0}")

    # Step 2: get_cohort_default_rate
    cohort_rate = get_cohort_default_rate(applicant)
    print(f"\n  [2] get_cohort_default_rate()")
    print(f"      Return value  : {cohort_rate}")
    print(f"      Type          : {type(cohort_rate).__name__}")
    print(f"      In range 0-100?: {0.0 <= cohort_rate <= 100.0}")

    # Step 3: explain_risk
    explanation = explain_risk(applicant, risk_score, cohort_rate)
    print(f"\n  [3] explain_risk()")
    print(f"      Return value : {explanation[:200]}{'...' if len(explanation) > 200 else ''}")
    print(f"      Type         : {type(explanation).__name__}")
    print(f"      Non-empty?   : {bool(explanation)}")

    # Small delay for Gemini rate limits
    time.sleep(4)

    # Step 4: recommend_decision
    decision = recommend_decision(explanation)
    print(f"\n  [4] recommend_decision()")
    print(f"      Return value : {decision}")
    print(f"      Type         : {type(decision).__name__}")
    print(f"      Has 'recommendation'?: {'recommendation' in decision}")
    print(f"      Has 'confidence'?    : {'confidence' in decision}")
    print(f"      recommendation value : {decision.get('recommendation')}")
    print(f"      confidence value     : {decision.get('confidence')}")

    # Validate expected types
    assert isinstance(risk_score, float), f"risk_score should be float, got {type(risk_score)}"
    assert isinstance(cohort_rate, float), f"cohort_rate should be float, got {type(cohort_rate)}"
    assert isinstance(explanation, str), f"explanation should be str, got {type(explanation)}"
    assert isinstance(decision, dict), f"decision should be dict, got {type(decision)}"
    assert "recommendation" in decision, "decision missing 'recommendation' key"
    assert "confidence" in decision, "decision missing 'confidence' key"

    print(f"\n  [OK] All assertions passed for {label}")


if __name__ == "__main__":
    print("=" * 60)
    print("  FULL-CHAIN INTEGRATION TEST")
    print("  score_applicant -> get_cohort_default_rate -> explain_risk -> recommend_decision")
    print("=" * 60)

    for applicant in SAMPLE_APPLICANTS:
        run_chain(dict(applicant))  # copy so .pop("label") doesn't mutate original
        time.sleep(4)  # respect Gemini rate limits between applicants

    print(f"\n{'='*60}")
    print("  [OK] ALL TESTS PASSED -- full chain is type-safe end to end")
    print(f"{'='*60}")


## Section 3 — Verify Files Exist on Disk
Run this cell after Section 2 to confirm every file was written correctly.

In [ ]:
# 3.1 — List all written source files
import os

expected_files = [
    'src/__init__.py',
    'src/person_a/__init__.py',
    'src/person_a/column_mapping.py',
    'src/person_a/generate_sample_data.py',
    'src/person_a/clean_data.py',
    'src/person_a/train.py',
    'src/person_a/scoring_service.py',
    'src/person_a/gpu_benchmark.py',
    'src/person_a/cloud_upload.py',
    # ── Person B files ──
    'src/person_b/__init__.py',
    'src/person_b/fake_applicants.py',
    'src/person_b/explain_api.py',
    'src/person_b/decision_engine.py',
    # ── Tests ──
    'tests/test_full_chain.py',
]

all_ok = True
for path in expected_files:
    exists = os.path.isfile(path)
    size   = os.path.getsize(path) if exists else 0
    tag    = '[OK]  ' if exists else '[MISS]'
    print(f'{tag} {path:<55} {size:>7} bytes')
    if not exists: all_ok = False

print()
print('[ALL OK] All files present.' if all_ok else '[ERROR] Some files are missing — re-run Section 2.')

In [ ]:
# 3.2 — Preview first 5 lines of each Person B file
import os
person_b_files = [
    'src/person_b/__init__.py',
    'src/person_b/fake_applicants.py',
    'src/person_b/explain_api.py',
    'src/person_b/decision_engine.py',
]
for path in person_b_files:
    print(f'\n{"="*60}')
    print(f'  FILE: {path}')
    print(f'{"="*60}')
    if os.path.isfile(path):
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= 5: break
                print(f'  {i+1}: {line}', end='')
    else:
        print('  [MISSING] — run Section 2B again')

## Section 4 — Core Pipeline: Data Generation, Cleaning & Model Training (Person A)

In [ ]:
# 4.1 — Generate synthetic loan applicant dataset
!python -u src/person_a/generate_sample_data.py

In [ ]:
# 4.2 — Clean and preprocess the data
!python -u src/person_a/clean_data.py

In [ ]:
# 4.3 — Train the RandomForest risk classification model
!python -u src/person_a/train.py

In [ ]:
# 4.4 — Verify model and data artifacts
import os
for f in ['data/cs-training.csv', 'data/clean_data.csv', 'data/model.pkl']:
    size = os.path.getsize(f) if os.path.isfile(f) else None
    status = f'{size/1e6:.2f} MB' if size else 'MISSING'
    print(f'  {f}: {status}')

## Section 5 — Google Cloud / BigQuery Upload
> **Skip this section** if you don't have a GCP project. The rest of the notebook works without it.

In [ ]:
# 5.1 — Authenticate with your Google Account
from google.colab import auth
auth.authenticate_user()
print('[OK] Authenticated!')

In [ ]:
# 5.2 — Set your GCP Project ID
import os
os.environ['GCP_PROJECT_ID'] = 'YOUR-PROJECT-ID-HERE'  # <-- CHANGE THIS
os.environ['BQ_DATASET']     = 'loan_underwriting'
os.environ['BQ_TABLE']       = 'loan_applicants'

In [ ]:
# 5.3 — Upload clean_data.csv to BigQuery
!python -u src/person_a/cloud_upload.py

## Section 6 — CPU vs GPU Performance Benchmark

In [ ]:
# 6.1 — Run feature-engineering benchmark (Pandas CPU vs cuDF GPU)
!python -u src/person_a/gpu_benchmark.py

In [ ]:
# 6.2 — Display the benchmark chart
from IPython.display import Image, display
import os
chart = 'data/cpu_vs_gpu_benchmark.png'
if os.path.isfile(chart):
    display(Image(chart))
else:
    print('[WARN] Chart not found — ensure benchmark completed successfully')

## Section 7 — Person B: GenAI Risk Explanations & Underwriting Decisions

### Prerequisites
1. Confirm Person B files exist by re-checking Section 3 output.
2. Confirm `model.pkl` and `clean_data.csv` were created in Section 4.
3. Add your **GEMINI_API_KEY** secret in the left sidebar 🔑 before running 7.1.

In [ ]:
# 7.1 — Load GEMINI_API_KEY from Colab Secrets
import os
from google.colab import userdata

try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    masked = os.environ['GEMINI_API_KEY'][:6] + '...' + os.environ['GEMINI_API_KEY'][-4:]
    print(f'[OK] GEMINI_API_KEY loaded: {masked}')
except Exception as e:
    print(f'[ERROR] Could not load GEMINI_API_KEY: {e}')
    print('        Add it via the key icon (🔑) in the Colab left sidebar.')

In [ ]:
# 7.2 — Verify Person B files are present before running
import os
files = {
    'src/person_b/fake_applicants.py':  'Applicant profiles',
    'src/person_b/explain_api.py':       'Gemini explanation engine',
    'src/person_b/decision_engine.py':   'Gemini decision engine',
}
ready = True
for path, label in files.items():
    ok = os.path.isfile(path)
    print(f"  {'[OK]  ' if ok else '[MISS]'} {path} — {label}")
    ready = ready and ok
print()
if ready:
    print('[READY] All Person B files confirmed. Proceed to 7.3.')
else:
    print('[ACTION NEEDED] Run Section 2B first to write the missing files.')

In [ ]:
# 7.3 — Run explain_api.py: generates Gemini natural-language risk explanations
#         for 8 fake applicants (Low / Medium / High risk)
!python -u src/person_b/explain_api.py

In [ ]:
# 7.4 — Run decision_engine.py: calls Gemini to produce structured JSON decisions
#         { recommendation: Approve | Manual Review | Decline,
#           confidence:      High | Medium | Low }
!python -u src/person_b/decision_engine.py

## Section 8 — End-to-End Integration Test
Chains **Person A scoring** → **BigQuery cohort rate** → **Person B Gemini explanation** → **Person B decision** for 2 applicants.

In [ ]:
# 8.1 — Run full-chain integration test
!python -u tests/test_full_chain.py

## Section 9 — Export Artifacts
Downloads `loan_underwriting_data.zip` to your computer. Extract it into your local `data/` folder to run the React frontend.

In [ ]:
# 9.1 — Precompute demo cohort cache
from src.person_a.scoring_service import precompute_demo_cache
precompute_demo_cache()

In [ ]:
# 9.2 — Zip and download all data artifacts
import shutil
from google.colab import files
shutil.make_archive('loan_underwriting_data', 'zip', 'data')
print('[OK] Zipped. Starting download...')
files.download('loan_underwriting_data.zip')